# ModernBERT Training & Evaluation

**Model:** `QuangDuy/bert-tiny-stage2-hf` (ModernBERT, max_length=4096)  
**Dataset:** `ContextSearchLM/context_search_vietnamese_prompt_224_minilmtok_finetune`  
**Train style:** 3 (hard-negative triplet: query + positive + negative)

## 1. Install & Import

In [ ]:
!pip install -q torch transformers datasets pyvi tqdm

In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from torch import nn
from typing import List, Dict
from tqdm import tqdm
import random

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Config

In [ ]:
# ── Model & Dataset ────────────────────────────────────────────────
MODEL_NAME = "QuangDuy/bert-tiny-stage2-hf"
DATASET_NAME = "ContextSearchLM/context_search_vietnamese_prompt_224_minilmtok_finetune"
MODEL_REPO_ID = "ContextSearchLM/modernbert_tiny_dropinfonce_prompt"  # push to hub

# ── HuggingFace token ──────────────────────────────────────────────
HF_TOKEN = None  # <- paste your token here or set to None

# ── Training hyperparameters ───────────────────────────────────────
MAX_LENGTH = 4096
BATCH_SIZE = 48
LEARNING_RATE = 5e-5
NUM_REPEAT = 1
TRAIN_STYLE = 3         # 3 = hard-negative triplet
LOSS_FN = "dropinfonce"  # "dropinfonce" or "infonce"
MODEL_TYPE = "mean_pooling"  # "mean_pooling" or "cls_pooling"
DROPOUT_LIST = (0.1, 0.15)
OUTPUT_DIR = "./modernbert_output"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Train style: {TRAIN_STYLE} (hard-negative triplet)")
print(f"Max length: {MAX_LENGTH}")

## 3. Loss Functions

In [ ]:
from torch.nn.functional import cosine_similarity as pairwise_cosine

def full_cosine_similarity(x, y):
    return torch.mm(x / x.norm(dim=-1)[:, None], (y / y.norm(dim=-1)[:, None]).transpose(0, 1))

def cosine_similarity_matrix(x, y):
    return full_cosine_similarity(x, y)

# ── Triplet losses ─────────────────────────────────────────────────

def unitdropinf(x, y, temperature=0.4):
    xy_mat = torch.exp(full_cosine_similarity(x, y) / temperature)
    prob = xy_mat.diagonal() / xy_mat.sum(dim=-1)
    return (-torch.log(prob) * (1 - prob)).mean()

def dropinfonceloss(x, y, z, temperature=0.4):
    if z is not None:
        xy_mat = torch.exp(pairwise_cosine(x, y) / temperature)
        xz_mat = torch.exp(pairwise_cosine(x, z) / temperature)
        prob = xy_mat / (xy_mat + xz_mat)
        return (-torch.log(prob) * (1 - prob)).mean()
    else:
        return (unitdropinf(x, y, temperature) + unitdropinf(y, x, temperature) +
                unitdropinf(x, x, temperature) + unitdropinf(y, y, temperature)) / 4

def infonceloss(x, y, z, temperature=0.4):
    if z is not None:
        xy_mat = torch.exp(pairwise_cosine(x, y) / temperature)
        xz_mat = torch.exp(pairwise_cosine(x, z) / temperature)
        return -torch.log(xy_mat / (xy_mat + xz_mat)).mean()
    else:
        xy_mat = torch.exp(full_cosine_similarity(x, y) / temperature)
        return -torch.log(xy_mat.diagonal() / xy_mat.sum(dim=-1)).mean()

LOSS_FUNCTIONS = {
    "dropinfonce": dropinfonceloss,
    "infonce": infonceloss,
}

print(f"Using loss: {LOSS_FN}")

## 4. Model Wrapper

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def modernbert_mean_pooling(model, input_ids, attention_mask, **kwargs):
    output = model(input_ids=input_ids, attention_mask=attention_mask)
    embeddings = mean_pooling(output, attention_mask)
    return F.normalize(embeddings, p=2, dim=1)

def modernbert_cls_pooling(model, input_ids, attention_mask, **kwargs):
    output = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
    return output.last_hidden_state[:, 0]

MODEL_WRAPPERS = {
    "mean_pooling": modernbert_mean_pooling,
    "cls_pooling": modernbert_cls_pooling,
}

model_wrapper = MODEL_WRAPPERS[MODEL_TYPE]
print(f"Pooling: {MODEL_TYPE}")

## 5. SimilarityLoss Model

In [ ]:
def change_dropout(model, p):
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.p = p

class SimilarityLoss(nn.Module):
    def __init__(self, model, model_wrapper, loss_fn, dropout_list=(0.1, 0.15)):
        super().__init__()
        self.model = model
        self.model_wrapper = model_wrapper
        self.loss_fn = loss_fn
        self.dropout_list = dropout_list

    def forward(self, query_ids, query_attention_mask,
                positive_ids=None, positive_attention_mask=None,
                negative_ids=None, negative_attention_mask=None,
                logger=False):
        change_dropout(self.model, self.dropout_list[0])
        query_emb = self.model_wrapper(
            self.model,
            query_ids.to(self.model.device),
            query_attention_mask.to(self.model.device)
        )

        if positive_ids is not None:
            pos_emb = self.model_wrapper(
                self.model,
                positive_ids.to(self.model.device),
                positive_attention_mask.to(self.model.device)
            )
            if negative_ids is not None:
                neg_emb = self.model_wrapper(
                    self.model,
                    negative_ids.to(self.model.device),
                    negative_attention_mask.to(self.model.device)
                )
            else:
                neg_emb = None
        else:
            change_dropout(self.model, self.dropout_list[1])
            pos_emb = self.model_wrapper(
                self.model,
                query_ids.to(self.model.device),
                query_attention_mask.to(self.model.device)
            )
            neg_emb = None

        output = (self.loss_fn(query_emb, pos_emb, neg_emb),)
        if logger:
            output += (query_emb, pos_emb, neg_emb)
        return output

    def push_to_hub(self, *args, **kwargs):
        self.model.push_to_hub(*args, **kwargs)

    def save_pretrained(self, *args, **kwargs):
        self.model.save_pretrained(*args, **kwargs)

## 6. Load Model & Tokenizer

In [ ]:
# Load base model
base_model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.padding_side = 'right'

# Add <|query|> special token if not present
if '<|query|>' not in tokenizer.get_vocab():
    print('Adding special token <|query|> to tokenizer')
    tokenizer.add_tokens('<|query|>')
    base_model.resize_token_embeddings(len(tokenizer))

print(f"Tokenizer:")
print(f"  [CLS] = {tokenizer.cls_token_id}")
print(f"  [SEP] = {tokenizer.sep_token_id}")
print(f"  [PAD] = {tokenizer.pad_token_id}")
print(f"  [MASK] = {tokenizer.mask_token_id}")
print(f"  Vocab size = {len(tokenizer)}")
print(f"  Max length = {tokenizer.model_max_length}")
print(f"\nModel: {type(base_model).__name__}")
print(f"Hidden size: {base_model.config.hidden_size}")
print(f"Num layers: {base_model.config.num_hidden_layers}")
print(f"Num heads: {base_model.config.num_attention_heads}")

# Wrap in SimilarityLoss
loss_fn = LOSS_FUNCTIONS[LOSS_FN]
model = SimilarityLoss(base_model, model_wrapper, loss_fn, DROPOUT_LIST)

# Push initial model to hub
if HF_TOKEN and MODEL_REPO_ID:
    base_model.push_to_hub(MODEL_REPO_ID, token=HF_TOKEN, private=True)
    tokenizer.push_to_hub(MODEL_REPO_ID, token=HF_TOKEN, private=True)
    print(f"Pushed to {MODEL_REPO_ID}")

## 7. Dataset (Re-tokenize from raw text)

Dataset có sẵn `query_ids`, `positive_ids`... nhưng đó là token IDs của **MiniLM**, không dùng được cho **ModernBERT**.  
→ Lấy raw text (`query`, `pos`, `neg`) rồi tokenize lại bằng ModernBERT tokenizer.

In [ ]:
class RetokenizedDataset(Dataset):
    """Load raw text from HF dataset, re-tokenize with target tokenizer."""

    def __init__(self, dataset, tokenizer, max_length=4096, include_negative=True):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.include_negative = include_negative

        # Filter empty
        self.dataset = dataset.filter(
            lambda x: x['query'] != '' and x['pos'] != '' and
            (not include_negative or x['neg'] != '')
        )
        print(f"Loaded {len(self.dataset)} samples (include_negative={include_negative})")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        q = self.tokenizer(item['query'], max_length=self.max_length,
                           truncation=True, padding=False, return_attention_mask=True)
        p = self.tokenizer(item['pos'], max_length=self.max_length,
                           truncation=True, padding=False, return_attention_mask=True)

        result = {
            'query_ids': q['input_ids'],
            'query_attention_mask': q['attention_mask'],
            'positive_ids': p['input_ids'],
            'positive_attention_mask': p['attention_mask'],
        }

        if self.include_negative:
            n = self.tokenizer(item['neg'], max_length=self.max_length,
                               truncation=True, padding=False, return_attention_mask=True)
            result['negative_ids'] = n['input_ids']
            result['negative_attention_mask'] = n['attention_mask']

        return result

In [ ]:
# Load dataset
print("Loading dataset...")
raw_train = load_dataset(DATASET_NAME, split='train', token=HF_TOKEN)
raw_eval = load_dataset(DATASET_NAME, split='validation', token=HF_TOKEN)

include_negative = (TRAIN_STYLE == 3)
print(f"\nTrain style {TRAIN_STYLE}: include_negative={include_negative}")

train_dataset = RetokenizedDataset(raw_train, tokenizer, MAX_LENGTH, include_negative)
eval_dataset = RetokenizedDataset(raw_eval, tokenizer, MAX_LENGTH, include_negative)

In [ ]:
# Verify tokenization — compare MiniLM IDs vs ModernBERT IDs
sample = raw_train[0]
print("=" * 60)
print("SAMPLE COMPARISON")
print("=" * 60)
print(f"Query text: {sample['query'][:80]}...")
print(f"\nMiniLM IDs (from dataset):   {sample['query_ids'][:15]}...")

new_enc = tokenizer(sample['query'], max_length=MAX_LENGTH, truncation=True)
print(f"ModernBERT IDs (re-tokenized): {new_enc['input_ids'][:15]}...")
print(f"\n[CLS] in dataset: {sample['query_ids'][0]} (MiniLM=101)")
print(f"[CLS] re-tokenized: {new_enc['input_ids'][0]} (ModernBERT=0) ✓")

## 8. Collator

In [ ]:
class ModernBertCollator:
    """Dynamic padding with correct pad_token_id for ModernBERT (pad=2)."""

    def __init__(self, pad_token_id=2):
        self.pad_token_id = pad_token_id

    def __call__(self, features: List[Dict]) -> Dict:
        batch = {}
        for key in features[0].keys():
            tensors = [torch.tensor(f[key]) for f in features]
            pad_value = self.pad_token_id if 'ids' in key else 0
            max_len = max(t.shape[0] for t in tensors)
            padded = []
            for t in tensors:
                pad_size = max_len - t.shape[0]
                if pad_size > 0:
                    t = F.pad(t, (0, pad_size), value=pad_value)
                padded.append(t.unsqueeze(0))
            batch[key] = torch.cat(padded, dim=0)
        return batch

collator = ModernBertCollator(pad_token_id=tokenizer.pad_token_id)
print(f"Collator pad_token_id = {tokenizer.pad_token_id}")

## 9. Evaluation Helper

In [ ]:
def get_batch_similar_info(similar):
    batch_size = similar.shape[0]
    data_pair = similar.diagonal()
    diff_pair = similar.sum() - data_pair.sum()
    return (float(data_pair.mean()),
            float(diff_pair / (batch_size * batch_size - batch_size)))

def check_model(model, eval_dataset, collator, eval_batch=64):
    eval_result = []
    with torch.no_grad():
        for i in tqdm(range(0, len(eval_dataset), eval_batch), desc="Evaluating"):
            batch = collator([eval_dataset[k] for k in range(i, min(i + eval_batch, len(eval_dataset)))])
            loss, query, pos, neg = model(**batch, logger=True)
            result = (float(loss),)
            result += get_batch_similar_info(cosine_similarity_matrix(query, pos))
            if neg is not None:
                result += get_batch_similar_info(cosine_similarity_matrix(query, neg))
            eval_result.append(result)
    eval_result = torch.tensor(eval_result).mean(dim=0).tolist()
    return {
        'description': f'loss={eval_result[0]:.4f}, pair_pos={eval_result[1]:.4f}, diff_pos={eval_result[2]:.4f}' +
                       (f', pair_neg={eval_result[3]:.4f}, diff_neg={eval_result[4]:.4f}' if len(eval_result) > 3 else ''),
        'loss': eval_result[0],
    }

## 10. Train! 🚀

In [ ]:
seed = random.randint(0, 100)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    hub_token=HF_TOKEN,
    do_train=True,
    seed=seed,
    logging_steps=1000,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=1,
    save_total_limit=1,
)

trainer = Trainer(
    args=training_args,
    processing_class=tokenizer,
    model=model,
    train_dataset=train_dataset,
    data_collator=collator,
)

best_result = float('inf')

for repeat_idx in range(NUM_REPEAT):
    print(f"\n{'='*60}")
    print(f"  Repeat {repeat_idx + 1}/{NUM_REPEAT}")
    print(f"{'='*60}")

    # Eval before training
    output_dict = check_model(model, eval_dataset, collator, eval_batch=BATCH_SIZE)
    print(f"Eval: {output_dict['description']}")

    if output_dict['loss'] < best_result:
        best_result = output_dict['loss']
        print(">> Better model found, pushing to hub...")
        if HF_TOKEN and MODEL_REPO_ID:
            model.push_to_hub(MODEL_REPO_ID, token=HF_TOKEN,
                              commit_message=output_dict['description'])

    # Train
    trainer.train()

# Final eval
print(f"\n{'='*60}")
print("  Final Evaluation")
print(f"{'='*60}")
output_dict = check_model(model, eval_dataset, collator, eval_batch=BATCH_SIZE)
print(f"Final: {output_dict['description']}")

if output_dict['loss'] < best_result:
    print(">> Better model found, pushing to hub...")
    if HF_TOKEN and MODEL_REPO_ID:
        model.push_to_hub(MODEL_REPO_ID, token=HF_TOKEN,
                          commit_message=output_dict['description'])

print("\n✅ Training complete!")

## 11. Evaluate on Benchmarks

In [ ]:
!pip install -q pyvi

In [ ]:
import numpy as np

class FlexiEmbedding:
    """Embedding model for evaluation — tokenizes from raw text."""

    def __init__(self, model_name, model_wrapper, token=None, device='cuda'):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
        self.tokenizer.padding_side = 'right'
        self.device = device
        self.model = AutoModel.from_pretrained(
            model_name, token=token, trust_remote_code=True
        ).to(device).eval()
        self.model_wrapper = model_wrapper

    def encode(self, sentences, max_length=512, **kwargs):
        batch_size = kwargs.get('batch_size', 32)
        with torch.no_grad():
            if isinstance(sentences[0], str):
                result = []
                for i in tqdm(range(0, len(sentences), batch_size), desc="Encoding"):
                    inputs = self.tokenizer(
                        sentences[i:i+batch_size], return_tensors="pt",
                        padding=True, max_length=max_length, truncation=True
                    ).to(self.device)
                    emb = self.model_wrapper(self.model, **inputs)
                    result.append(emb)
                return torch.concat(result, dim=0).cpu()
            else:
                result = []
                for group in sentences:
                    if len(group) == 0:
                        result.append(None)
                    else:
                        inputs = self.tokenizer(
                            group, return_tensors="pt",
                            padding=True, max_length=max_length, truncation=True
                        ).to(self.device)
                        emb = self.model_wrapper(self.model, **inputs)
                        result.append(emb)
                return result

### 11a. Evaluate ViGLUE-R (Reranking)

In [ ]:
EVAL_MODEL = MODEL_REPO_ID if HF_TOKEN else MODEL_NAME

eval_model = FlexiEmbedding(
    model_name=EVAL_MODEL,
    model_wrapper=model_wrapper,
    token=HF_TOKEN,
    device=DEVICE
)

# ── ViGLUE-R: MNLI-R ──────────────────────────────────────────────
print("\n" + "="*60)
print("ViGLUE-R: MNLI-R")
print("="*60)

viglue_mnli = load_dataset('ContextSearchLM/ViGLUE-R', split='mnli_r', token=HF_TOKEN)

query_emb = eval_model.encode(viglue_mnli['anchor'])
pos_emb = eval_model.encode(viglue_mnli['pos'])
neg_emb = eval_model.encode(viglue_mnli['neg'])

pos_sim = pairwise_cosine(query_emb, pos_emb)
neg_sim = pairwise_cosine(query_emb, neg_emb)
accuracy = (pos_sim > neg_sim).float().mean().item()
print(f"MNLI-R Accuracy: {accuracy:.4f}")

# ── ViGLUE-R: QNLI-R ──────────────────────────────────────────────
print("\n" + "="*60)
print("ViGLUE-R: QNLI-R")
print("="*60)

viglue_qnli = load_dataset('ContextSearchLM/ViGLUE-R', split='qnli_r', token=HF_TOKEN)

query_emb = eval_model.encode(viglue_qnli['anchor'])
pos_emb = eval_model.encode(viglue_qnli['pos'])
neg_emb = eval_model.encode(viglue_qnli['neg'])

pos_sim = pairwise_cosine(query_emb, pos_emb)
neg_sim = pairwise_cosine(query_emb, neg_emb)
accuracy = (pos_sim > neg_sim).float().mean().item()
print(f"QNLI-R Accuracy: {accuracy:.4f}")

### 11b. Evaluate ViNLI (Reranking)

In [ ]:
print("\n" + "="*60)
print("ViNLI Reranking")
print("="*60)

vinli = load_dataset('ContextSearchLM/ViNLI_reranking', split='test', token=HF_TOKEN)

query_emb = eval_model.encode(vinli['anchor'])
pos_emb = eval_model.encode(vinli['pos'])
neg_emb = eval_model.encode(vinli['neg'])

pos_sim = pairwise_cosine(query_emb, pos_emb)
neg_sim = pairwise_cosine(query_emb, neg_emb)
accuracy = (pos_sim > neg_sim).float().mean().item()
print(f"ViNLI Accuracy: {accuracy:.4f}")

### 11c. Evaluate ViMedAQA (Retrieval)

In [ ]:
from pyvi.ViTokenizer import tokenize as vi_tokenize

def text_split(text, text_max_length=96, text_duplicate=32):
    words = vi_tokenize(text).replace('_', ' ').split(' ')
    jump = text_max_length - text_duplicate
    return [' '.join(words[i*jump:i*jump+text_max_length]) for i in range(len(words)//jump + 1)]

for subset_name in ['all', 'drug', 'medicine', 'disease', 'body-part']:
    print(f"\n{'='*60}")
    print(f"ViMedAQA: {subset_name}")
    print(f"{'='*60}")

    vimedaqa = load_dataset('tmnam20/ViMedAQA', split='test', name=subset_name, token=HF_TOKEN)
    df = vimedaqa.to_pandas()[['question', 'context']].groupby('context').agg({
        'question': lambda x: list(x),
    }).reset_index()
    df = df[df['context'].str.strip() != '']
    df['context_chunks'] = df['context'].apply(text_split)

    context_idx, query_idx, query_list, context_list = [], [], [], []
    for i, row in df.iterrows():
        chunks = row['context_chunks']
        questions = [f'<|query|> {q}' for q in row['question']]
        context_idx += [i] * len(chunks)
        query_idx += [i] * len(questions)
        query_list += questions
        context_list += chunks

    labels = (torch.tensor([query_idx]).T == torch.tensor([context_idx])).float()

    query_emb = eval_model.encode(query_list)
    context_emb = eval_model.encode(context_list)

    sim_matrix = torch.mm(
        query_emb / query_emb.norm(dim=-1, keepdim=True),
        (context_emb / context_emb.norm(dim=-1, keepdim=True)).T
    )

    # MRR@10
    topk = sim_matrix.topk(min(10, sim_matrix.shape[1]), dim=1).indices
    mrr = 0.0
    for q_i in range(len(query_list)):
        for rank, idx in enumerate(topk[q_i]):
            if labels[q_i, idx] > 0:
                mrr += 1.0 / (rank + 1)
                break
    mrr /= len(query_list)

    # Recall@10
    recall = 0.0
    for q_i in range(len(query_list)):
        relevant = labels[q_i].sum().item()
        if relevant > 0:
            hits = sum(1 for idx in topk[q_i] if labels[q_i, idx] > 0)
            recall += hits / relevant
    recall /= len(query_list)

    print(f"  MRR@10:    {mrr:.4f}")
    print(f"  Recall@10: {recall:.4f}")

## Done! ✅